# Volume-based Gini — Alpha Cash Stack Model (Train)

Volume counterpart of `calculate_periodic_gini_prod_ver_trench_dimfact`. Segmentation, weekly/monthly periods, `segment_type` naming and the fact/dimension structure are unchanged; only the Gini itself is volume-weighted.

**Method.** Each loan carries two masses instead of a 0/1 label:
- bad volume = `def*_v` (outstanding principal at the due date if the loan defaulted)
- good volume = `flg_mature_*_v` − `def*_v` (disbursed amount minus the bad part)

AUC = probability that a random unit of bad volume scores higher than a random unit of good volume (ties = 0.5); Gini = 2·AUC − 1. With 0/1 weights it reduces exactly to the count-based Gini.

**New fact-table columns:** `bad_volume`, `total_volume` (bad rate by volume = `bad_volume / total_volume`). They change the schema, so this notebook writes to **new** BigQuery tables.

**Before running:** `client` (BigQuery client) and `update_tables(...)` must already exist in the session, as in your current notebook.

In [ ]:
import warnings
import numpy as np
import pandas as pd
from itertools import combinations
from google.cloud import bigquery

## 1. Function

In [ ]:
def calculate_gini_volume_fast(scores, bad_vol, good_vol):
    """
    Volume-weighted Gini via a weighted Mann-Whitney AUC.

    Every account carries two masses instead of one 0/1 label:
        bad_vol  -> its bad volume   (e.g. outstanding principal at the due date if defaulted)
        good_vol -> its good volume  (total_volume - bad_volume)
    AUC = P(a random unit of BAD volume has a higher score than a random unit of
    GOOD volume), ties counted as 0.5. Gini = 2 * AUC - 1.

    With bad_vol = label (0/1) and good_vol = 1 - label this reduces exactly to
    calculate_gini_fast (the count-based version).

    Returns np.nan for fewer than 2 rows, or when bad volume or good volume sums to zero.
    """
    n = scores.shape[0]
    if n < 2:
        return np.nan
    total_bad = bad_vol.sum()
    total_good = good_vol.sum()
    if total_bad <= 0 or total_good <= 0:
        return np.nan

    # collapse to unique score values; tied scores share one bucket
    _, inv = np.unique(scores, return_inverse=True)
    bad_u = np.bincount(inv, weights=bad_vol)
    good_u = np.bincount(inv, weights=good_vol)

    # good volume strictly below each score + half of the good volume tied at it
    good_below = np.cumsum(good_u) - good_u
    auc = (bad_u * (good_below + 0.5 * good_u)).sum() / (total_bad * total_good)
    return 2.0 * auc - 1.0


def calculate_periodic_gini_prod_ver_trench_dimfact_volume(
    df,
    score_column,
    bad_volume_column,
    total_volume_column,
    namecolumn,
    data_selection_column=None,
    model_version_column=None,
    trench_column=None,
    loan_type_column=None,
    loan_product_type_column=None,
    ostype_column=None,
    apptype_column=None,
    risk_segment_column=None,
    risk_segment_final_column=None,
    account_id_column=None,
    bad_flag_column=None,
):
    """
    Volume-based counterpart of calculate_periodic_gini_prod_ver_trench_dimfact.

    Segmentation logic is unchanged: same power set of segment columns, same
    weekly/monthly grains, same segment_type naming, same fact/dimension
    structure and segment_id assignment. What changes is the Gini itself:

    - bad_volume_column   : per-account bad volume   (e.g. deffpd0_v)
    - total_volume_column : per-account total volume (e.g. flg_mature_fpd0_v)
    - good volume = total_volume - bad_volume (floored at 0)
    - gini_value is the volume-weighted Gini (see calculate_gini_volume_fast)

    Extra fact-table columns: bad_volume and total_volume (sums per segment/period),
    so bad rate by volume = bad_volume / total_volume.

    distinct_accounts / bad_count keep their count-based meaning. If
    bad_flag_column (e.g. 'deffpd0') is given, bad_count = distinct accounts with
    flag == 1 (matches the count-based tables); otherwise bad_count = distinct
    accounts with bad_volume > 0.
    """
    required_columns = ["disbursementdate", score_column, bad_volume_column, total_volume_column]
    if not all(col in df.columns for col in required_columns):
        raise ValueError(f"Missing required columns. Need: {required_columns}")

    optional_columns = {
        "data_selection": data_selection_column,
        "model_version": model_version_column,
        "trench": trench_column,
        "loan_type": loan_type_column,
        "loan_product_type": loan_product_type_column,
        "ostype": ostype_column,
        "apptype": apptype_column,
        "risk_segment": risk_segment_column,
        "risk_segment_final": risk_segment_final_column,
        "account_id": account_id_column,
        "bad_flag": bad_flag_column,
    }
    for col_name, col in optional_columns.items():
        if col and col not in df.columns:
            raise ValueError(
                f"{col_name.replace('_', ' ').title()} column '{col}' not found in dataframe"
            )

    df = df.copy()
    df["disbursementdate"] = pd.to_datetime(df["disbursementdate"])
    df[score_column] = pd.to_numeric(df[score_column], errors="coerce")
    df[bad_volume_column] = pd.to_numeric(df[bad_volume_column], errors="coerce")
    df[total_volume_column] = pd.to_numeric(df[total_volume_column], errors="coerce")

    rows_before = len(df)
    df = df.dropna(subset=[score_column, bad_volume_column, total_volume_column])
    n_dropped = rows_before - len(df)
    if n_dropped:
        warnings.warn(
            f"{n_dropped} row(s) dropped for NULL score / bad volume / total volume "
            f"(e.g. a defaulted loan with no outstanding-principal record)."
        )

    # precompute both period grains ONCE for the whole frame
    df["week"] = df["disbursementdate"].dt.to_period("W")
    df["month"] = df["disbursementdate"].dt.to_period("M")

    # (display_name_for_segment_type_string, source_column, output_field_name)
    segment_defs = []
    if data_selection_column:
        segment_defs.append(("DataSelection", data_selection_column, "data_selection"))
    if model_version_column:
        segment_defs.append(("ModelVersion", model_version_column, "model_version"))
    if trench_column:
        segment_defs.append(("Trench", trench_column, "trench_category"))
    if loan_type_column:
        segment_defs.append(("LoanType", loan_type_column, "loan_type"))
    if loan_product_type_column:
        segment_defs.append(("ProductType", loan_product_type_column, "loan_product_type"))
    if ostype_column:
        segment_defs.append(("OSType", ostype_column, "ostype"))
    if apptype_column:
        segment_defs.append(("apptype", apptype_column, "apptype"))
    if risk_segment_column:
        segment_defs.append(("risk_segment", risk_segment_column, "risk_segment"))
    if risk_segment_final_column:
        segment_defs.append(("risk_segment_final", risk_segment_final_column, "risk_segment_final"))

    all_output_fields = [
        "data_selection", "model_version", "trench_category", "loan_type",
        "loan_product_type", "ostype", "apptype", "risk_segment", "risk_segment_final",
    ]

    scores_arr = df[score_column].to_numpy()
    bad_arr = np.clip(df[bad_volume_column].to_numpy(dtype=float), 0.0, None)
    total_arr = np.clip(df[total_volume_column].to_numpy(dtype=float), 0.0, None)
    n_over = int((bad_arr > total_arr).sum())
    if n_over:
        warnings.warn(
            f"{n_over} row(s) have bad volume > total volume; their good volume is floored at 0."
        )
    good_arr = np.clip(total_arr - bad_arr, 0.0, None)

    accounts_arr = df[account_id_column].to_numpy() if account_id_column else None
    if account_id_column:
        if bad_flag_column:
            bad_acct_arr = pd.to_numeric(df[bad_flag_column], errors="coerce").to_numpy() == 1
        else:
            bad_acct_arr = bad_arr > 0

    # build the list of column-subsets to process: r=0 ("Overall") .. full power set
    combos = [()]
    for r in range(1, len(segment_defs) + 1):
        combos.extend(combinations(segment_defs, r))

    period_specs = [
        ("week", "Week", lambda ts: ts, lambda ts: ts + pd.Timedelta(days=6)),
        ("month", "Month", lambda ts: ts, lambda ts: ts + pd.DateOffset(months=1) - pd.Timedelta(days=1)),
    ]

    rows = []
    for combo in combos:
        combo_source_cols = [col for _, col, _ in combo]
        for period_col, period_label, start_fn, end_fn in period_specs:
            key_cols = combo_source_cols + [period_col]

            gb = df.groupby(key_cols, observed=True, dropna=True, sort=True)
            group_indices = gb.indices  # {key: positional row-index array}

            for key, idx in group_indices.items():
                if not isinstance(key, tuple):
                    key = (key,)
                seg_values = key[:-1]
                period_value = key[-1]

                seg_scores = scores_arr[idx]
                seg_bad = bad_arr[idx]
                seg_good = good_arr[idx]
                gini_value = calculate_gini_volume_fast(seg_scores, seg_bad, seg_good)

                bad_volume = float(seg_bad.sum())
                total_volume = float(total_arr[idx].sum())

                if account_id_column:
                    seg_accounts = accounts_arr[idx]
                    distinct_accounts = pd.unique(seg_accounts).shape[0]
                    bad_mask = bad_acct_arr[idx]
                    bad_count = (
                        pd.unique(seg_accounts[bad_mask]).shape[0] if bad_mask.any() else 0
                    )
                else:
                    distinct_accounts = None
                    bad_count = None

                start_date = start_fn(period_value.to_timestamp())
                end_date = end_fn(period_value.to_timestamp())

                seg_name = (
                    "Overall"
                    if not combo
                    else "_".join(f"{disp}_{val}" for (disp, _, _), val in zip(combo, seg_values))
                )

                row = {
                    "start_date": start_date,
                    "end_date": end_date,
                    "period": period_label,
                    "gini_value": gini_value,
                    "distinct_accounts": distinct_accounts,
                    "bad_count": bad_count,
                    "bad_volume": bad_volume,
                    "total_volume": total_volume,
                    "Model_Name": score_column,
                    "bad_rate": namecolumn,
                    "segment_type": seg_name,
                }
                for field in all_output_fields:
                    row[field] = None
                for (_, _, field), val in zip(combo, seg_values):
                    row[field] = val

                rows.append(row)

    fact_table = pd.DataFrame(rows)
    fact_table = fact_table.sort_values(["segment_type", "start_date"]).reset_index(drop=True)

    dimension_table = (
        fact_table[["Model_Name", "bad_rate", "segment_type"] + all_output_fields]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    dimension_table["segment_id"] = range(len(dimension_table))

    fact_table = fact_table.merge(
        dimension_table[["segment_id", "Model_Name", "bad_rate", "segment_type"] + all_output_fields],
        on=["Model_Name", "bad_rate", "segment_type"] + all_output_fields,
        how="left",
    )

    fact_table = fact_table[
        ["segment_id", "start_date", "end_date", "period", "gini_value", "distinct_accounts",
         "bad_count", "bad_volume", "total_volume", "Model_Name", "bad_rate", "segment_type"]
        + all_output_fields
    ]
    dimension_table = dimension_table[
        ["segment_id", "Model_Name", "bad_rate", "segment_type"] + all_output_fields
    ]

    return fact_table, dimension_table

## 2. Check the function (no BigQuery needed)

1. Weighted Gini equals sklearn's AUC with `sample_weight` (each loan split into a bad row and a good row).
2. With 0/1 weights it equals the count-based rank Gini.
3. If the count-based `calculate_periodic_gini_prod_ver_trench_dimfact` is defined in this session, a unit-weight run must reproduce its `gini_value`, `distinct_accounts` and `bad_count` exactly.

In [ ]:
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(7)
n = 2000
chk = pd.DataFrame({
    "acct": np.arange(n),
    "disbursementdate": pd.Timestamp("2026-01-01") + pd.to_timedelta(rng.integers(0, 90, n), unit="D"),
    "grp": rng.choice(["a", "b"], n),
    "os": rng.choice(["android", "ios"], n),
})
chk["score"] = np.round(rng.normal(size=n), 1)                     # coarse -> many ties
chk["bad"] = (rng.random(n) < 1 / (1 + np.exp(-(chk["score"] * 0.9 - 2)))).astype(int)
chk["tot_v"] = rng.choice([5000.0, 10000.0, 20000.0], n)
chk["bad_v"] = np.where(chk["bad"] == 1, chk["tot_v"] * rng.uniform(0.6, 1.0, n), 0.0)

s, b, t = chk["score"].to_numpy(), chk["bad_v"].to_numpy(), chk["tot_v"].to_numpy()

# 1. vs sklearn weighted AUC
ref = 2 * roc_auc_score(np.r_[np.ones(n), np.zeros(n)], np.r_[s, s], sample_weight=np.r_[b, t - b]) - 1
mine = calculate_gini_volume_fast(s, b, t - b)
print(f"1. weighted Gini {mine:.12f} | sklearn {ref:.12f} | diff {abs(mine - ref):.2e}")

# 2. unit weights vs rank-based count Gini
lab = chk["bad"].to_numpy().astype(float)
r = rankdata(s, method="average"); npos = lab.sum(); nneg = n - npos
count_gini = 2 * ((r[lab == 1].sum() - npos * (npos + 1) / 2) / (npos * nneg)) - 1
print(f"2. unit-weight diff vs count Gini: {abs(calculate_gini_volume_fast(s, lab, 1 - lab) - count_gini):.2e}")

# 3. full function vs the count-based function (if available in this session)
seg_kw = dict(data_selection_column="grp", ostype_column="os", account_id_column="acct")
chk["one"] = 1.0
f_v, d_v = calculate_periodic_gini_prod_ver_trench_dimfact_volume(
    chk, "score", "bad", "one", "CHK", bad_flag_column="bad", **seg_kw)
if "calculate_periodic_gini_prod_ver_trench_dimfact" in globals():
    f_c, d_c = calculate_periodic_gini_prod_ver_trench_dimfact(chk, "score", "bad", "CHK", **seg_kw)
    cols = list(f_c.columns)
    print("3. max gini diff:", np.nanmax((f_c["gini_value"] - f_v[cols]["gini_value"]).abs()),
          "| row/count mismatches:",
          int((f_c[["segment_id", "distinct_accounts", "bad_count"]].values
               != f_v[["segment_id", "distinct_accounts", "bad_count"]].values).sum()),
          "| dimension identical:", d_c.equals(d_v))
else:
    print("3. skipped (count-based function not defined in this session)")

## 3. Input data

Table IDs are new (extra volume columns). Model: `Alpha-Cash-Stack-Model` / `alpha_stack_model_cash`, score column `aStackScore`.

In [ ]:
# ---- sanity: the two objects from your existing notebook must exist ----
assert "client" in globals(), "define client = bigquery.Client(project='prj-prod-dataplatform') first"
assert "update_tables" in globals(), "define/import your existing update_tables helper first"

facttable_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.fact_alphastackcash_volume_train"
dimtable_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.dimension_alphastackcash_volume_train"

In [ ]:
sq = """
WITH parsed as (
select mmrd.customerId, mmrd.digitalLoanAccountId,modelDisplayName,modelVersionId,start_time,end_time,prediction,trenchCategory,
REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeatures,
Data_selection,     deviceOs osType,

from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116 mmrd
      where modelDisplayName in ('Alpha-Cash-Stack-Model', 'alpha_stack_model_cash')
),
  modelname as (
  select customerId,
  digitalLoanAccountId,
  start_time,
  prediction aStackScore,
  case when p.modelDisplayName like 'Alpha%' then 'alpha_stack_model_cash' else p.modelDisplayName end as modelDisplayName,
  p.modelVersionId,
  trenchCategory,
  case when trenchCategory in ('Trench 1', 'Trench 2') then 'New_Applicant' else 'Repeat_Applicant' end Application_type,
  Data_selection,
       osType,
  from parsed p
  )
  ,
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
  segmentdata AS (
    SELECT
      loan.customerid,
      loan.digitalLoanAccountId,
      trench_category.trenchCategory,
      loan.offer_id,
      CASE
        WHEN COALESCE(trench1_seg.risk_segment) IS NULL
          THEN 'Unsegmented'
        ELSE COALESCE(trench1_seg.risk_segment)
        END AS risk_segment,
      appVersion,
      flagApproval,
      tsa_onboarding_time,
      IF(
        applicationStatus IN ('COMPLETED', 'ACTIVATED', 'APPROVED'),
        'Loan Approved',
        'Loan Not Approved') AS loan_application_status,
      -- if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
      DATE(decision_date) AS application_date
    FROM
      (
        SELECT DISTINCT
          digitalLoanAccountId,
          customerId,
          applicationStatus,
          disbursementDateTime,
          date(decision_date) decision_date,
          appVersion,
          flagApproval,
          tsa_onboarding_time,
          offer_id
        FROM `risk_credit_mis.loan_master_table`
        WHERE
          date(decision_date) >= date('2025-11-10') AND new_loan_type = 'Quick'
        -- QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
      ) loan
    LEFT JOIN
      (
        SELECT
          digitalLoanAccountId,
          CASE
            WHEN trenchCategory = 'Trench 1' THEN 'Trench-1'
            WHEN trenchCategory = 'Trench 2' THEN 'Trench-2'
            WHEN trenchCategory = 'Trench 3' THEN 'Trench-3'
            END AS trenchCategory,
          publish_time
        FROM `audit_balance.ml_model_run_details`
        WHERE
          modelDisplayName in ('Alpha-Cash-Stack-Model', 'alpha_stack_model_cash')
        QUALIFY
          row_number()
            OVER (PARTITION BY digitalLoanAccountId ORDER BY publish_time DESC)
          = 1
      ) trench_category
      ON trench_category.digitalLoanAccountId = loan.digitalLoanAccountId
    LEFT JOIN
      (
        SELECT
          cust_id, risk_segment, created_date, created_by, offer_id
        FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
        WHERE offer_type = 'SEGMENTED_ACL'
        -- AND created_by='GCP-API-CALL'
        -- QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
      ) trench1_seg
      ON trench1_seg.offer_id = loan.offer_id
  ),
base as
(select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  r.aStackScore,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month,
  Data_selection,
  del.deffpd0,
  case when del.deffpd0 = 1 then (select Total_Outstanding_Principal FROM prj-prod-dataplatform.risk_credit_mis.loan_bucket_flow_report_core 
                                                                      where loanAccountNumber = loanmaster.loanAccountNumber 
                                                                      and date(sourceDataAsOf) = date(loanmaster.firstduedate)) else 0 end as deffpd0_v,
  del.flg_mature_fpd0,
  case when del.flg_mature_fpd0 = 1 then loanmaster.disbursedLoanAmount else 0 end flg_mature_fpd0_v,
  del.deffpd10,
  case when del.deffpd10 = 1 then (select Total_Outstanding_Principal FROM prj-prod-dataplatform.risk_credit_mis.loan_bucket_flow_report_core 
                                                                      where loanAccountNumber = loanmaster.loanAccountNumber 
                                                                      and date(sourceDataAsOf) = date_add(date(loanmaster.firstduedate), interval 10 day)) else 0 end as deffpd10_v,
  del.flg_mature_fpd10,
  case when del.flg_mature_fpd10 = 1 then loanmaster.disbursedLoanAmount else 0 end flg_mature_fpd10_v,
  del.deffpd30,
  case when del.deffpd30 = 1 then (select Total_Outstanding_Principal FROM prj-prod-dataplatform.risk_credit_mis.loan_bucket_flow_report_core 
                                                                      where loanAccountNumber = loanmaster.loanAccountNumber 
                                                                      and date(sourceDataAsOf) = date_add(date(loanmaster.firstduedate), interval 30 day)) else 0 end as deffpd30_v,
  del.flg_mature_fpd30,
  case when del.flg_mature_fpd30 = 1 then loanmaster.disbursedLoanAmount else 0 end flg_mature_fpd30_v,
  del.deffspd30,
  case when del.deffspd30 = 1 then (select Total_Outstanding_Principal FROM prj-prod-dataplatform.risk_credit_mis.loan_bucket_flow_report_core 
                                                                      where loanAccountNumber = loanmaster.loanAccountNumber 
                                                                      and date(sourceDataAsOf) = date_add(date(loanmaster.firstduedate), interval 60 day)) else 0 end as deffspd30_v,
  del.flg_mature_fspd_30,
  case when del.flg_mature_fspd_30 = 1 then loanmaster.disbursedLoanAmount else 0 end flg_mature_fspd_30_v,
  del.deffstpd30,
  case when del.deffstpd30 = 1 then (select Total_Outstanding_Principal FROM prj-prod-dataplatform.risk_credit_mis.loan_bucket_flow_report_core 
                                                                      where loanAccountNumber = loanmaster.loanAccountNumber 
                                                                      and date(sourceDataAsOf) = date_add(date(loanmaster.firstduedate), interval 90 day)) else 0 end as deffstpd30_v,
  del.flg_mature_fstpd_30,
  case when del.flg_mature_fstpd_30 = 1 then loanmaster.disbursedLoanAmount else 0 end flg_mature_fstpd_30_v,
  loanmaster.new_loan_type,
  modelVersionId,
    r.trenchCategory,
    r.Application_type,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile'
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall'
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,

    coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end),
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
    
    coalesce(sd.risk_segment, 'NA') risk_segment,
    coalesce(frs.risk_segment_final, 'NA') risk_segment_final
    from modelname r
  left join `risk_credit_mis.loan_master_table` loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  inner join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
  left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
   LEFT JOIN segmentdata sd
      ON sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
    LEFT JOIN
      (
        SELECT digitalLoanAccountid, risk_segment_final
        FROM prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response
        WHERE risk_segment_final IS NOT NULL
        QUALIFY
          row_number()
            OVER (PARTITION BY digitalLoanAccountid ORDER BY created_dt DESC)
          = 1
      ) frs
      ON frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and r.aStackScore is not null
  and del.flg_mature_fpd0 = 1
  )
  select * from base
  qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
  ;
"""
dfd = client.query(sq).to_dataframe()
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()

**Reconciliation before calculating.** For each metric: defaulted loans, defaulted loans whose bad volume is NULL or 0 (NULL rows are dropped by the function with a warning; zeros count as good volume), and the bad rate by count vs by volume.

In [ ]:
_metrics = [
    ("FPD0", "deffpd0", "deffpd0_v", "flg_mature_fpd0", "flg_mature_fpd0_v"),
    ("FPD10", "deffpd10", "deffpd10_v", "flg_mature_fpd10", "flg_mature_fpd10_v"),
    ("FPD30", "deffpd30", "deffpd30_v", "flg_mature_fpd30", "flg_mature_fpd30_v"),
    ("FSPD30", "deffspd30", "deffspd30_v", "flg_mature_fspd_30", "flg_mature_fspd_30_v"),
    ("FSTPD30", "deffstpd30", "deffstpd30_v", "flg_mature_fstpd_30", "flg_mature_fstpd_30_v"),
]
_rows = []
for name, flag, bad_v, mat, mat_v in _metrics:
    m = dfd[dfd[mat] == 1]
    bv = pd.to_numeric(m[bad_v], errors="coerce")
    tv = pd.to_numeric(m[mat_v], errors="coerce")
    d = m[flag] == 1
    _rows.append({
        "metric": name, "mature_loans": len(m), "defaulted": int(d.sum()),
        "defaulted_bad_v_null": int((d & bv.isna()).sum()),
        "defaulted_bad_v_zero": int((d & (bv == 0)).sum()),
        "bad_rate_count": d.mean(), "bad_rate_volume": bv.sum() / tv.sum(),
    })
pd.DataFrame(_rows)

## FPD0
### Train

In [ ]:
df_concat = dfd.copy()

In [ ]:
df_concat["aStackScore"] = pd.to_numeric(
    df_concat["aStackScore"], errors="coerce"
)
print("Volume Gini Calculation for Alpha cash stack score FPD0 started")

In [ ]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact_volume(
    df_concat,
    "aStackScore",
    "deffpd0_v",
    "flg_mature_fpd0_v",
    "FPD0",
    data_selection_column="Data_selection",
    model_version_column="modelVersionId",
    trench_column="trenchCategory",
    loan_type_column="new_loan_type",
    loan_product_type_column="loan_product_type",
    ostype_column="osType",
    apptype_column="Application_type",
    risk_segment_column="risk_segment",
    risk_segment_final_column="risk_segment_final",
    account_id_column="digitalLoanAccountId",
    bad_flag_column="deffpd0",
)

In [ ]:
fact_table, dimension_table = update_tables(
    fact_table, dimension_table, model_name="alpha_stack_model_cash", product="CASH"
)
print(f"The shape of the fact table is:\t {fact_table.shape}")
print(f"The shape of the dimension table is:\t {dimension_table.shape}")

df_f_fpd0_alphastackcash_v = fact_table.copy()
df_d_fpd0_alphastackcash_v = dimension_table.copy()

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
)
job = client.load_table_from_dataframe(df_f_fpd0_alphastackcash_v, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

In [ ]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
)
job = client.load_table_from_dataframe(
    df_d_fpd0_alphastackcash_v, dimtable_id, job_config=job_config
)
job.result()  # Wait for the job to complete

## FPD10
### Train

In [ ]:
print(f"The shape of the dataframe downloaded fpd10 is:\t {dfd[dfd['flg_mature_fpd10']==1].shape}")
dfd[dfd['flg_mature_fpd10']==1].head()

In [ ]:
df_concat = dfd[dfd['flg_mature_fpd10']==1].copy()

In [ ]:
df_concat["aStackScore"] = pd.to_numeric(
    df_concat["aStackScore"], errors="coerce"
)
print("Volume Gini Calculation for Alpha cash stack score FPD10 started")

In [ ]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact_volume(
    df_concat,
    "aStackScore",
    "deffpd10_v",
    "flg_mature_fpd10_v",
    "FPD10",
    data_selection_column="Data_selection",
    model_version_column="modelVersionId",
    trench_column="trenchCategory",
    loan_type_column="new_loan_type",
    loan_product_type_column="loan_product_type",
    ostype_column="osType",
    apptype_column="Application_type",
    risk_segment_column="risk_segment",
    risk_segment_final_column="risk_segment_final",
    account_id_column="digitalLoanAccountId",
    bad_flag_column="deffpd10",
)

In [ ]:
fact_table, dimension_table = update_tables(
    fact_table, dimension_table, model_name="alpha_stack_model_cash", product="CASH"
)
print(f"The shape of the fact table is:\t {fact_table.shape}")
print(f"The shape of the dimension table is:\t {dimension_table.shape}")

df_f_fpd10_alphastackcash_v = fact_table.copy()
df_d_fpd10_alphastackcash_v = dimension_table.copy()

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",
)
job = client.load_table_from_dataframe(df_f_fpd10_alphastackcash_v, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

In [ ]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",
)
job = client.load_table_from_dataframe(
    df_d_fpd10_alphastackcash_v, dimtable_id, job_config=job_config
)
job.result()  # Wait for the job to complete

## FPD30
### Train

In [ ]:
print(f"The shape of the dataframe downloaded fpd30 is:\t {dfd[dfd['flg_mature_fpd30']==1].shape}")
dfd[dfd['flg_mature_fpd30']==1].head()

In [ ]:
df_concat = dfd[dfd['flg_mature_fpd30']==1].copy()

In [ ]:
df_concat["aStackScore"] = pd.to_numeric(
    df_concat["aStackScore"], errors="coerce"
)
print("Volume Gini Calculation for Alpha cash stack score FPD30 started")

In [ ]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact_volume(
    df_concat,
    "aStackScore",
    "deffpd30_v",
    "flg_mature_fpd30_v",
    "FPD30",
    data_selection_column="Data_selection",
    model_version_column="modelVersionId",
    trench_column="trenchCategory",
    loan_type_column="new_loan_type",
    loan_product_type_column="loan_product_type",
    ostype_column="osType",
    apptype_column="Application_type",
    risk_segment_column="risk_segment",
    risk_segment_final_column="risk_segment_final",
    account_id_column="digitalLoanAccountId",
    bad_flag_column="deffpd30",
)

In [ ]:
fact_table, dimension_table = update_tables(
    fact_table, dimension_table, model_name="alpha_stack_model_cash", product="CASH"
)
print(f"The shape of the fact table is:\t {fact_table.shape}")
print(f"The shape of the dimension table is:\t {dimension_table.shape}")

df_f_fpd30_alphastackcash_v = fact_table.copy()
df_d_fpd30_alphastackcash_v = dimension_table.copy()

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",
)
job = client.load_table_from_dataframe(df_f_fpd30_alphastackcash_v, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

In [ ]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",
)
job = client.load_table_from_dataframe(
    df_d_fpd30_alphastackcash_v, dimtable_id, job_config=job_config
)
job.result()  # Wait for the job to complete

## FSPD30
### Train

In [ ]:
print(f"The shape of the dataframe downloaded fspd30 is:\t {dfd[dfd['flg_mature_fspd_30']==1].shape}")
dfd[dfd['flg_mature_fspd_30']==1].head()

In [ ]:
df_concat = dfd[dfd['flg_mature_fspd_30']==1].copy()

In [ ]:
df_concat["aStackScore"] = pd.to_numeric(
    df_concat["aStackScore"], errors="coerce"
)
print("Volume Gini Calculation for Alpha cash stack score FSPD30 started")

In [ ]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact_volume(
    df_concat,
    "aStackScore",
    "deffspd30_v",
    "flg_mature_fspd_30_v",
    "FSPD30",
    data_selection_column="Data_selection",
    model_version_column="modelVersionId",
    trench_column="trenchCategory",
    loan_type_column="new_loan_type",
    loan_product_type_column="loan_product_type",
    ostype_column="osType",
    apptype_column="Application_type",
    risk_segment_column="risk_segment",
    risk_segment_final_column="risk_segment_final",
    account_id_column="digitalLoanAccountId",
    bad_flag_column="deffspd30",
)

In [ ]:
fact_table, dimension_table = update_tables(
    fact_table, dimension_table, model_name="alpha_stack_model_cash", product="CASH"
)
print(f"The shape of the fact table is:\t {fact_table.shape}")
print(f"The shape of the dimension table is:\t {dimension_table.shape}")

df_f_fspd30_alphastackcash_v = fact_table.copy()
df_d_fspd30_alphastackcash_v = dimension_table.copy()

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",
)
job = client.load_table_from_dataframe(df_f_fspd30_alphastackcash_v, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

In [ ]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",
)
job = client.load_table_from_dataframe(
    df_d_fspd30_alphastackcash_v, dimtable_id, job_config=job_config
)
job.result()  # Wait for the job to complete

## FSTPD30
### Train

In [ ]:
print(f"The shape of the dataframe downloaded fstpd30 is:\t {dfd[dfd['flg_mature_fstpd_30']==1].shape}")
dfd[dfd['flg_mature_fstpd_30']==1].head()

In [ ]:
df_concat = dfd[dfd['flg_mature_fstpd_30']==1].copy()

In [ ]:
df_concat["aStackScore"] = pd.to_numeric(
    df_concat["aStackScore"], errors="coerce"
)
print("Volume Gini Calculation for Alpha cash stack score FSTPD30 started")

In [ ]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact_volume(
    df_concat,
    "aStackScore",
    "deffstpd30_v",
    "flg_mature_fstpd_30_v",
    "FSTPD30",
    data_selection_column="Data_selection",
    model_version_column="modelVersionId",
    trench_column="trenchCategory",
    loan_type_column="new_loan_type",
    loan_product_type_column="loan_product_type",
    ostype_column="osType",
    apptype_column="Application_type",
    risk_segment_column="risk_segment",
    risk_segment_final_column="risk_segment_final",
    account_id_column="digitalLoanAccountId",
    bad_flag_column="deffstpd30",
)

In [ ]:
fact_table, dimension_table = update_tables(
    fact_table, dimension_table, model_name="alpha_stack_model_cash", product="CASH"
)
print(f"The shape of the fact table is:\t {fact_table.shape}")
print(f"The shape of the dimension table is:\t {dimension_table.shape}")

df_f_fstpd30_alphastackcash_v = fact_table.copy()
df_d_fstpd30_alphastackcash_v = dimension_table.copy()

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",
)
job = client.load_table_from_dataframe(df_f_fstpd30_alphastackcash_v, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

In [ ]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",
)
job = client.load_table_from_dataframe(
    df_d_fstpd30_alphastackcash_v, dimtable_id, job_config=job_config
)
job.result()  # Wait for the job to complete

## Combine

In [ ]:
factalphastackcash_v = pd.concat([df_f_fpd0_alphastackcash_v, df_f_fpd10_alphastackcash_v, df_f_fpd30_alphastackcash_v, df_f_fspd30_alphastackcash_v, df_f_fstpd30_alphastackcash_v], ignore_index=True)
dimalphastackcash_v = pd.concat([df_d_fpd0_alphastackcash_v, df_d_fpd10_alphastackcash_v, df_d_fpd30_alphastackcash_v, df_d_fspd30_alphastackcash_v, df_d_fstpd30_alphastackcash_v], ignore_index=True)
print(factalphastackcash_v.shape, dimalphastackcash_v.shape)